In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

import os 
import pickle
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
data = data.drop(columns=['EmployeeCount', 'StandardHours', 'EmployeeNumber', 'Over18'])

print(data.shape)
data.head(10)

(1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2
5,32,No,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,4,Male,79,3,1,Laboratory Technician,4,Single,3068,11864,0,No,13,3,3,0,8,2,2,7,7,3,6
6,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,3,Female,81,4,1,Laboratory Technician,1,Married,2670,9964,4,Yes,20,4,1,3,12,3,2,1,0,0,0
7,30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,4,Male,67,3,1,Laboratory Technician,3,Divorced,2693,13335,1,No,22,4,2,1,1,2,3,1,0,0,0
8,38,No,Travel_Frequently,216,Research & Development,23,3,Life Sciences,4,Male,44,2,3,Manufacturing Director,3,Single,9526,8787,0,No,21,4,2,0,10,2,3,9,7,1,8
9,36,No,Travel_Rarely,1299,Research & Development,27,3,Medical,3,Male,94,3,2,Healthcare Representative,3,Married,5237,16577,6,No,13,3,2,2,17,3,2,7,7,7,7


In [3]:
# Encode target first, separately
target = 'Attrition'

# Separate feature types
categorical_cols = data.select_dtypes(include='object').columns.tolist()
numerical_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target from categorical list
categorical_cols.remove(target)

print("Categorical columns:", categorical_cols)
print("\nNumerical columns:", numerical_cols)
print(f"\nTotal: {len(categorical_cols)} categorical, {len(numerical_cols)} numerical")

Categorical columns: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Numerical columns: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Total: 7 categorical, 23 numerical


### Understanding Columns

- Binary Categories: Gender, OverTime
- Nominal Categories: BusinessTravel, Department, EducationField, JobRole, MaritalStatus
- Numerical Ordinal Categories: Education, EnvironmentSatisfaction, JobInvolvement, JobLevel, JobSatisfaction, PerformanceRating, RelationshipSatisfaction, StockOptionLevel, WorkLifeBalance

In [4]:
le_target = LabelEncoder()
data['Attrition'] = le_target.fit_transform(data['Attrition'])

print('Target Encoding: ', dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))
print("Value counts:\n", data['Attrition'].value_counts())

Target Encoding:  {'No': np.int64(0), 'Yes': np.int64(1)}
Value counts:
 Attrition
0    1233
1     237
Name: count, dtype: int64


In [5]:
binary_cols = ['Gender', 'OverTime']

le = LabelEncoder()

for col in binary_cols:
    data[col] = le.fit_transform(data[col])
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

Gender: {'Female': np.int64(0), 'Male': np.int64(1)}
OverTime: {'No': np.int64(0), 'Yes': np.int64(1)}


In [6]:
nominal_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']

data = pd.get_dummies(data, columns=nominal_cols, drop_first=True)

print("Shape after one-hot encoding:", data.shape)
print("New columns added:", [c for c in data.columns if any(nom in c for nom in nominal_cols)])

Shape after one-hot encoding: (1470, 45)
New columns added: ['BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely', 'Department_Research & Development', 'Department_Sales', 'EducationField_Life Sciences', 'EducationField_Marketing', 'EducationField_Medical', 'EducationField_Other', 'EducationField_Technical Degree', 'JobRole_Human Resources', 'JobRole_Laboratory Technician', 'JobRole_Manager', 'JobRole_Manufacturing Director', 'JobRole_Research Director', 'JobRole_Research Scientist', 'JobRole_Sales Executive', 'JobRole_Sales Representative', 'MaritalStatus_Married', 'MaritalStatus_Single']


In [7]:
print('Nulls: ', data.isnull().sum().sum())
print('Remaining Objects: ', data.select_dtypes(include='object').columns.tolist())
print('\nData Types Summary: ')
print(data.dtypes.value_counts())
data.head(3)

Nulls:  0
Remaining Objects:  []

Data Types Summary: 
int64    26
bool     19
Name: count, dtype: int64


,Age,Attrition,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single
0,41,1,1102,1,2,2,0,94,3,2,4,5993,19479,8,1,11,3,1,0,8,0,1,6,4,0,5,False,True,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
1,49,0,279,8,1,3,1,61,2,2,2,5130,24907,1,0,23,4,4,1,10,3,3,10,7,1,7,True,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False
2,37,1,1373,2,2,4,1,92,2,1,3,2090,2396,6,1,15,3,2,0,7,3,3,0,0,0,0,False,True,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True


In [8]:
x = data.drop(columns=['Attrition'])
y = data['Attrition']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print("Training set:", x_train.shape, "| Attrition rate:", round(y_train.mean() * 100, 2), "%")
print("Test set:", x_test.shape, "| Attrition rate:", round(y_test.mean() * 100, 2), "%")

Training set: (1176, 44) | Attrition rate: 16.16 %
Test set: (294, 44) | Attrition rate: 15.99 %


In [9]:
# Columns to scale — all numerical except one-hot encoded cols (already 0/1)
# We scale everything that isn't binary (0/1 already)
cols_to_scale = ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

scaler = StandardScaler()

# Fit ONLY on training data, then transform both
x_train[cols_to_scale] = scaler.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler.transform(x_test[cols_to_scale])  # <-- transform only, no fit

print("Scaling complete.")
print("MonthlyIncome mean after scaling (train):", round(x_train['MonthlyIncome'].mean(), 4))
print("MonthlyIncome std after scaling (train):", round(x_train['MonthlyIncome'].std(), 4))

Scaling complete.
MonthlyIncome mean after scaling (train): -0.0
MonthlyIncome std after scaling (train): 1.0004


In [10]:
print("Before SMOTE:")
print(f"  Training samples: {x_train.shape[0]}")
print(f"  Class distribution: {dict(pd.Series(y_train).value_counts().sort_index())}")

smote = SMOTE(random_state=42)
x_train_sm, y_train_sm = smote.fit_resample(x_train, y_train)

print("\nAfter SMOTE:")
print(f"  Training samples: {x_train_sm.shape[0]}")
print(f"  Class distribution: {dict(pd.Series(y_train_sm).value_counts().sort_index())}")
print(f"  Attrition rate: {round(y_train_sm.mean() * 100, 2)}%")

print("\nTest set unchanged:")
print(f"  Test samples: {x_test.shape[0]}")
print(f"  Class distribution: {dict(pd.Series(y_test).value_counts().sort_index())}")

Before SMOTE:
  Training samples: 1176
  Class distribution: {0: np.int64(986), 1: np.int64(190)}

After SMOTE:
  Training samples: 1972
  Class distribution: {0: np.int64(986), 1: np.int64(986)}
  Attrition rate: 50.0%

Test set unchanged:
  Test samples: 294
  Class distribution: {0: np.int64(247), 1: np.int64(47)}


In [11]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs/models', exist_ok=True)

# Save processed dataframes
x_train_sm.to_csv('../data/processed/x_train.csv', index=False)
x_test.to_csv('../data/processed/x_test.csv', index=False)
pd.Series(y_train_sm, name='Attrition').to_csv('../data/processed/y_train.csv', index=False)
pd.Series(y_test, name='Attrition').to_csv('../data/processed/y_test.csv', index=False)

# Save the scaler — critical for production use
with open('../outputs/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Saved:")
print("  data/processed/x_train.csv")
print("  data/processed/x_test.csv")
print("  data/processed/y_train.csv")
print("  data/processed/y_test.csv")
print("  outputs/models/scaler.pkl")

Saved:
  data/processed/x_train.csv
  data/processed/x_test.csv
  data/processed/y_train.csv
  data/processed/y_test.csv
  outputs/models/scaler.pkl


In [12]:
print("=" * 62)
print("       PREPROCESSING SUMMARY — 02_preprocessing.ipynb")
print("=" * 62)

print("""
ENCODING
  Binary (Label Encoded):   Gender, OverTime
  Nominal (One-Hot):        BusinessTravel, Department, EducationField, JobRole, MaritalStatus
  Target:                   Attrition → No=0, Yes=1

SHAPE
  Raw input:                (1470, 31)
  After encoding:           (1470, 45)
  After split:              Train (1176, 44) | Test (294, 44)

SCALING
  Method:                   StandardScaler (mean=0, std=1)
  Fitted on:                Training data only
  Applied to:               23 numerical columns

SMOTE
  Applied to:               Training data only
  Before:                   986 No | 190 Yes (1176 total)
  After:                    986 No | 986 Yes (1972 total)
  Test set:                 247 No | 47 Yes  (294 total, unchanged)

SAVED Training and Testing Sets and Scaler
""")
print("=" * 33)
print("  Ready for: 03_modeling.ipynb")
print("=" * 33)

       PREPROCESSING SUMMARY — 02_preprocessing.ipynb

ENCODING
  Binary (Label Encoded):   Gender, OverTime
  Nominal (One-Hot):        BusinessTravel, Department, EducationField, JobRole, MaritalStatus
  Target:                   Attrition → No=0, Yes=1

SHAPE
  Raw input:                (1470, 31)
  After encoding:           (1470, 45)
  After split:              Train (1176, 44) | Test (294, 44)

SCALING
  Method:                   StandardScaler (mean=0, std=1)
  Fitted on:                Training data only
  Applied to:               23 numerical columns

SMOTE
  Applied to:               Training data only
  Before:                   986 No | 190 Yes (1176 total)
  After:                    986 No | 986 Yes (1972 total)
  Test set:                 247 No | 47 Yes  (294 total, unchanged)

SAVED Training and Testing Sets and Scaler

  Ready for: 03_modeling.ipynb
